In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


In [ ]:
# Load master parquet file with merged NYISO load and weather data
from pathlib import Path

MASTER_PATH = Path(r"c:/Users/Matt/Desktop/CS506/CS506_Project/1_LIB/master/master.parquet")
print(f"Loading data from: {MASTER_PATH}")

# Load the master parquet file
df = pd.read_parquet(MASTER_PATH)

# Standardize datetime column name
if 'datetime' in df.columns:
    df['Time'] = pd.to_datetime(df['datetime'], utc=True)
elif 'Time Stamp' in df.columns:
    df['Time'] = pd.to_datetime(df['Time Stamp'], utc=True)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")


In [ ]:
# Weather features are already in master parquet
# Aggregate weather features by timestamp (average across stations/PTIDs)
weather_cols = [
    'temp_2m [degF]',
    'apparent_temperature [degF]',
    'relative_humidity [percent]',
    'precip_1hr [inch]',
    'avg_wind_speed_merge [mile/hr]',
    'solar_insolation [W/m^2]'
]

# Check which weather columns exist
existing_weather_cols = [col for col in weather_cols if col in df.columns]
print(f"Available weather features: {existing_weather_cols}")

# Aggregate weather data by timestamp
agg_dict = {col: 'mean' for col in existing_weather_cols}
agg_dict['Load'] = 'sum'

df_aggregated = df.groupby('Time', as_index=False).agg(agg_dict)
print(f"Aggregated data shape: {df_aggregated.shape}")
print(df_aggregated.head())


In [ ]:
# Rename columns to match expected format
rename_dict = {
    'temp_2m [degF]': 'avg_temp',
    'apparent_temperature [degF]': 'avg_apparent_temp',
    'relative_humidity [percent]': 'avg_humidity',
    'precip_1hr [inch]': 'total_precip',
    'avg_wind_speed_merge [mile/hr]': 'avg_wind_speed',
    'solar_insolation [W/m^2]': 'avg_solar'
}

df_final = df_aggregated.rename(columns=rename_dict)
print(df_final.head())
print(f"\nFinal dataset shape: {df_final.shape}")


In [ ]:
# Resample to 5-MINUTE intervals (highest resolution)
df_final['Time'] = pd.to_datetime(df_final['Time'])
df_final = df_final.set_index('Time')

df_5min = df_final.resample('5min').mean().reset_index()
df_5min = df_5min.dropna()

print(f"5-minute aggregated data shape: {df_5min.shape}")
print(df_5min.head())

# Data is ready to use
merged = df_5min.copy()
df_total_load = merged


In [ ]:
# Split the data based on the year
train_data = df_total_load[df_total_load['Time'].dt.year.between(2001, 2021)]
val_data = df_total_load[df_total_load['Time'].dt.year == 2022]
test_data = df_total_load[df_total_load['Time'].dt.year.isin([2023, 2024, 2025])]

# Print the sizes of each split
print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Testing data size: {len(test_data)}")

In [ ]:
train_data.head()

In [ ]:
train_data = train_data.dropna()
val_data = val_data.dropna()
test_data = test_data.dropna()
scaler = StandardScaler()
y_scaler = StandardScaler()

# Drop Time and Load columns for features
feature_cols = [col for col in train_data.columns if col not in ['Time', 'Load', 'index']]
print(f"Feature columns: {feature_cols}")

train_scaled = scaler.fit_transform(train_data[feature_cols])
val_scaled = scaler.transform(val_data[feature_cols])
test_scaled = scaler.transform(test_data[feature_cols])

# Use ravel() to convert to 1D array for scikit-learn compatibility
y_train_scaled = y_scaler.fit_transform(train_data[['Load']]).ravel()
y_val_scaled = y_scaler.transform(val_data[['Load']]).ravel()
y_test_scaled = y_scaler.transform(test_data[['Load']]).ravel()


In [ ]:
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        v = X[i:i + time_steps]
        Xs.append(v)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 5
X_train, y_train = create_dataset(train_scaled, y_train_scaled, TIME_STEPS)
X_val, y_val = create_dataset(val_scaled, y_val_scaled, TIME_STEPS)
X_test, y_test = create_dataset(test_scaled, y_test_scaled, TIME_STEPS)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

In [ ]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_val:", np.isnan(X_val).sum())
print("NaNs in y_train:", np.isnan(y_train).sum())
print("NaNs in y_val:", np.isnan(y_val).sum())

In [ ]:
# For 5-minute data (large dataset), use subset for tuning
subset_size = min(40000, len(X_train))
val_subset = min(8000, len(X_val))

X_train_sub = X_train[:subset_size]
y_train_sub = y_train[:subset_size]
X_val_sub = X_val[:val_subset]
y_val_sub = y_val[:val_subset]

print(f"Using {subset_size} training samples and {val_subset} validation samples for tuning")


In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
# best_mae = float('inf')
# best_model = None

# for C in [0.1, 1, 10]:
#     for gamma in ['scale', 0.01, 0.001]:
#         for epsilon in [0.01, 0.1, 0.5, 1.0]:
#             print(f"C={C}, gamma={gamma}, epsilon={epsilon}")
#             model = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon)
#             model.fit(X_train_sub.reshape(X_train_sub.shape[0], -1), y_train_sub)
#             preds = model.predict(X_val_sub.reshape(X_val_sub.shape[0], -1))
#             mae = mean_absolute_error(y_val_sub, preds)
#             print(f"val MAE={mae:.3f}")

#             if mae < best_mae:
#                 best_mae = mae
#                 best_model = model

# print("Best params found:", best_model.get_params())


Best params found: {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}

Best mae: 0.04994650252799576

**Note**: 5-minute data is the highest resolution. Training will take longest (~60-120 min) but provides best granularity.

In [ ]:
best_params = {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}
best_model = SVR(**best_params)
print(f"Training on 5-minute data ({len(X_train)} samples - this will take a while)...")
best_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
print("✓ Training complete!")

In [ ]:
# Make predictions
train_pred = best_model.predict(X_train.reshape(X_train.shape[0], -1))
train_pred = y_scaler.inverse_transform(train_pred.reshape(-1, 1))
y_train_inv = y_scaler.inverse_transform(y_train.reshape(-1, 1))

val_pred = best_model.predict(X_val.reshape(X_val.shape[0], -1))
val_pred = y_scaler.inverse_transform(val_pred.reshape(-1, 1))
y_val_inv = y_scaler.inverse_transform(y_val.reshape(-1, 1))

test_pred = best_model.predict(X_test.reshape(X_test.shape[0], -1))
test_pred = y_scaler.inverse_transform(test_pred.reshape(-1, 1))
y_test_inv = y_scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:
# Evaluate the model
mae_train = mean_absolute_error(y_train_inv, train_pred)
mae_test = mean_absolute_error(y_test_inv, test_pred)
print("Mean Absolute Error on Training Data:", mae_train)
print("Mean Absolute Error on Testing Data:", mae_test)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

mape_train = mean_absolute_percentage_error(y_train_inv, train_pred)
mape_test = mean_absolute_percentage_error(y_test_inv, test_pred)
print("Mean Absolute Percentage Error on Training Data:", mape_train)
print("Mean Absolute Percentage Error on Testing Data:", mape_test)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100 

y_true = y_test_inv
y_pred = test_pred
x = np.arange(len(y_true))

window = 100
lag = 1

fig, ax = plt.subplots(figsize=(10,6))
line_true, = ax.plot([], [], label="True Values", color="blue", alpha=0.7)
line_pred, = ax.plot([], [], label="Predictions", color="red", alpha=0.7)
ax.set_ylim(min(y_true.min(), y_pred.min())*0.95, max(y_true.max(), y_pred.max())*1.05)
ax.set_xlabel("Index")
ax.set_ylabel("Load")
ax.set_title("SVR Predictions vs True Load (5-Minute Resolution)")
ax.legend()

def update(frame):
    start = max(0, frame - window)
    end = frame
    line_true.set_data(x[start:end], y_true[start:end])
    
    pred_start = max(0, frame - window - lag)
    pred_end = max(0, frame - lag)
    line_pred.set_data(x[pred_start:pred_end], y_pred[pred_start:pred_end])
    
    ax.set_xlim(x[start], x[end-1] if end > start else x[start]+1)
    return line_true, line_pred

ani = FuncAnimation(
    fig, update,
    frames=range(0, len(x), 10),
    interval=20, blit=True
)
HTML(ani.to_jshtml())


In [ ]:
mape = np.mean(np.abs((y_test_inv - test_pred) / y_test_inv)) * 100
print(f"Testing MAPE: {mape:.2f}%")

eps = 1e-6
mape = np.mean(np.abs((y_train_inv - train_pred) / (y_train_inv + eps))) * 100
print(f"Training MAPE: {mape:.2f}%")

mape = np.mean(np.abs((y_val_inv - val_pred) / y_val_inv)) * 100
print(f"Val MAPE: {mape:.2f}%")


In [ ]:
from sklearn.metrics import r2_score

rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred))
print(f"Testing RMSE: {rmse}")

r2 = r2_score(y_test_inv, test_pred)
print(f"Testing R2: {r2}")

In [ ]:
import joblib

joblib.dump(best_model, "svr_5minute_model.joblib")
print("Model saved as: svr_5minute_model.joblib")


In [ ]:
subset_start = 0
subset_end = 1000
plt.figure(figsize=(12,6))
plt.plot(y_test_inv[subset_start:subset_end], label="True Load", color="black", alpha=0.7)
plt.plot(test_pred[subset_start:subset_end], label="Predictions", color="orange", alpha=0.7)
plt.xlabel("Time Index (5-minute intervals)")
plt.ylabel("Load (MW)")
plt.title("SVR Predictions vs True Load (5-Minute Resolution)")
plt.legend()
plt.show()